# Gyokusai: top-down walkthrough

Gyokusai prepares web-scale text for model pretraining. It is the data-processing layer, not the model-training loop: source-specific loaders and small Daft components are composed into the pipeline needed for each corpus.

The runnable examples below use synthetic data and stay offline. Model downloads, GPUs, external services, and large filesystem jobs are shown only as explicit boundaries.

## System map

```text
files and datasets
  -> loaders and checkpoints
  -> WARC, HTML, math, or OCR extraction
  -> encoding cleanup and row filters
  -> corpus-level deduplication
  -> language, PII, embeddings, generation, and model scores
  -> review, mixing, and versioned clean-text outputs
  -> BlitzBERT tokenization, packing, and pretraining
```

Most components return lazy Daft DataFrames. Work starts when a result is collected or written. `DataEngine` is a small sequential composer; it does not impose schemas or a fixed pipeline.

In [ ]:
RUN_EXTERNAL_DATA = False
RUN_MODEL_STAGES = False
RUN_BATCH_JOBS = False

import daft

from gyokusai.encoding import FixEncoding
from gyokusai.engine import DataEngine
from gyokusai.filters.c4.dedupe import dedupe_3_sentences
from gyokusai.filters.filters import LengthFilter, LoremIpsumFilter
from gyokusai.filters.url_blacklist import URLBlacklistFilter
from gyokusai.parsers.html import ParseHtml

## 1. Load data and establish a page schema

`loader.py` provides Parquet, JSON, CSV, WARC, Lance, and Hugging Face readers. File readers can use Daft checkpoints so repeated jobs skip source objects already processed. `WarcLoader` produces raw WARC columns; `ExtractWarc` keeps response records and emits the common page fields `warc_hash`, `html`, `url`, `record_id`, `crawl_date`, and `source_path`.

The synthetic input starts at that post-WARC boundary.

In [ ]:
if RUN_EXTERNAL_DATA:
    from gyokusai.loader import WarcLoader
    from gyokusai.parsers.warc import ExtractWarc

    warc_records = WarcLoader(
        checkpoint_path="/shared/checkpoints/warc",
        num_workers=16,
        cpus_per_worker=1,
    ).read_data("/shared/common-crawl/*.warc.gz")
    raw_pages = ExtractWarc()(warc_records)

In [ ]:
def article_html(title: str, sentence: str) -> str:
    body = sentence * 10
    return f"<!doctype html><html><head><title>{title}</title></head><body><main><h1>{title}</h1><p>{body}</p></main></body></html>"


raw_pages = daft.from_pydict(
    {
        "record_id": ["keep-1", "blocked-1", "lorem-1"],
        "url": [
            "https://docs.example/guide",
            "https://ads.example/page",
            "https://docs.example/placeholder",
        ],
        "html": [
            article_html(
                "Guide", "A useful corpus keeps provenance and readable cafÃ© text. "
            ),
            article_html(
                "Advertisement",
                "Commercial navigation should leave during URL filtering. ",
            ),
            article_html(
                "Placeholder",
                "Lorem ipsum material should leave during heuristic filtering. ",
            ),
        ],
        "source_path": ["sample.warc"] * 3,
    }
)

## 2. Compose row-level preparation

`ParseHtml` turns HTML into text with Resiliparse or Trafilatura. `FixEncoding` repairs mojibake. Filters either remove rows or rewrite the text column; the current set covers URL domains, length, bad words, lorem ipsum, JavaScript lines, minimum lines or words, braces, and punctuation ratios.

The blacklist is supplied in memory here so the example does not download the repository's default list.

In [ ]:
prepare = DataEngine(
    components=[
        URLBlacklistFilter(blacklist={"ads.example"}),
        ParseHtml(parser_type="trafilatura"),
        FixEncoding(),
        LengthFilter(min_len=80),
        LoremIpsumFilter(),
    ],
    name="page_preparation",
)
prepared_pages = prepare.run(raw_pages).collect()

prepared = prepared_pages.to_pydict()
assert prepared["record_id"] == ["keep-1"]
assert "café" in prepared["text"][0]
prepared_pages.select("record_id", "url", "text")

## 3. Run corpus-level operations separately

`dedupe_3_sentences` is an in-memory DataFrame operation that drops documents whose three-sentence spans are all repeats. `fuzzy_dedupe` is a larger MinHash/LSH job: it switches to the Ray runner and reads and writes checkpointed Parquet directories. That path-oriented lifecycle is kept outside the row-level `DataEngine`.

In [ ]:
shared = "First sentence. Second sentence. Third sentence."
dedupe_input = daft.from_pydict(
    {
        "record_id": ["a", "b", "c"],
        "text": [shared, shared, "Fourth sentence. Fifth sentence. Sixth sentence."],
    }
)
deduplicated = dedupe_3_sentences(dedupe_input).collect().sort("record_id")

assert deduplicated.to_pydict()["record_id"] == ["a", "c"]
deduplicated

In [ ]:
if RUN_BATCH_JOBS:
    from gyokusai.deduplication.deduplicate import fuzzy_dedupe

    fuzzy_dedupe(
        input_path="/shared/corpus/pages",
        work="/shared/work/fuzzy-dedupe",
        output_dir="/shared/corpus/deduplicated",
        key="record_id",
        text="text",
    )

## 4. Add specialized or model-backed stages

| Component | Data effect | Runtime boundary |
|---|---|---|
| `ParseMath` | HTML to math-preserving text | Requires `w3m` or `lynx` |
| `OCRText` | Page image to Markdown | vLLM and GPU |
| `ExtractLanguage` | Adds `language` | Downloads GlotLID/FastText |
| `PIIAnonymizer` | Rewrites `text` | Presidio models |
| `EmbedText` | Adds `text_embedding` | SentenceTransformer and GPU |
| `GenerateText` | Adds a generated extraction | SGLang and GPU |
| `AIContentScorer` and domain report | Adds a score and review candidates; filtering remains separate | Hugging Face classifier on CPU or GPU |
| `TokenCounter` / `TokenizeText` | Adds counts or token IDs | Downloads the selected tokenizer |

The repository also contains a Stack Exchange archive-to-Parquet pipeline, persona-conditioned query generation, math extraction and grading prompts, and JSON schemas for math topics and educational level. `parsers/pdf.py` is currently only a placeholder.

In [ ]:
if RUN_MODEL_STAGES:
    from gyokusai.factories.ai_content import AIContentScorer
    from gyokusai.factories.embedding import EmbedText
    from gyokusai.factories.generation import GenerateText
    from gyokusai.filters.tokenize import TokenCounter
    from gyokusai.language_id.lang_id import ExtractLanguage
    from gyokusai.pii.pii import PIIAnonymizer

    language_pages = ExtractLanguage()(prepared_pages)
    masked_pages = PIIAnonymizer()(language_pages)
    counted_pages = masked_pages.with_column(
        "num_tokens", TokenCounter().num_tokens(daft.col("text"))
    )
    embedded_pages = EmbedText(model_name="lightonai/DenseOn")(masked_pages)
    generated_pages = GenerateText(model_path="organization/extraction-model")(
        masked_pages
    )
    scored_pages = AIContentScorer(
        model_name="organization/validated-detector",
        revision="full-model-revision",
        ai_label="generated",
        gpus=1,
    )(masked_pages)

## 5. Persist clean text and hand off

Gyokusai should publish clean `text` with stable IDs, provenance, policy decisions, and any features needed for data mixing. Its tokenizer utilities are optional; they are not the current BlitzBERT interface.

[`BlitzBERT/prepare_data/map_dataset.py`](https://github.com/teraflop-ai/BlitzBERT/blob/34eb821e2105e2ef0a7720a0bb12040a10613ea8/prepare_data/map_dataset.py) loads a Hugging Face dataset containing `text`, applies the configured tokenizer, packs documents to a context length, and saves `input_ids` with document `offsets`. `MappedDataset` then loads that packed artifact for training. There is not yet a direct Gyokusai-to-BlitzBERT adapter or canonical publication schema.

In [ ]:
clean_text_pages = prepared_pages.select("record_id", "url", "text", "source_path")
clean_text_pages.collect()

# Gyokusai production boundary:
# clean_text_pages.write_parquet("/shared/corpus/version=...")

## Current state

Gyokusai already has the main composable building blocks for web extraction, cleanup, filtering, deduplication, enrichment, and specialized datasets. Important seams remain: `parsers/pdf.py` is empty; the Stack Exchange path expects a missing `to_text` helper; output manifests and the BlitzBERT adapter are undefined; and the included SLURM scripts contain site-specific paths rather than a portable HPC recipe. Model-backed filters still need validation on representative data.

The downstream training path is also still evolving. The [current BlitzBERT entry point](https://github.com/teraflop-ai/BlitzBERT/blob/34eb821e2105e2ef0a7720a0bb12040a10613ea8/main.py) wires `BlitzDecoder` with its autoregressive packed collator. An [MLM collator](https://github.com/teraflop-ai/BlitzBERT/blob/34eb821e2105e2ef0a7720a0bb12040a10613ea8/src/blitzbert/data.py#L70-L159) exists, but the encoder-style masked-language-model path is not yet connected to the training entry point.